### Build an SVM machine using scikit-learn Pipeline

In [ ]:
dach_length = [55, 57, 64, 63, 58, 49, 54, 61]
dach_height = [30, 31, 36, 30, 33, 25, 37, 34]
jin_length = [56, 47, 56, 46, 49, 53, 52, 48]
jin_height = [52, 52, 50, 53, 50, 53, 49, 54]

In [ ]:
import numpy as np

d = np.column_stack((dach_length, dach_height))
j = np.column_stack((jin_length, jin_height))
X = np.concatenate((d, j))   # 데이터 집합
y = [0]*len(d) + [1]*len(j)  # 레이블 집합
print('dogs :', X)
print('labels :', y)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

svm = Pipeline([  # standard scaler, linear SVM classifier
               ('scaler', StandardScaler()),
               ('linearSVC', LinearSVC(C=1, loss='hinge', dual=True))
])
svm.fit(X, y)     # learn a svm model with X, y imputs

In [ ]:
dog_classes = {0:'Dachshund', 1:'Jindo dog'}
data1, data2 = [59, 35], [53, 54]
y_pred = svm.predict([data1])
print('data :', data1, ', classification:', dog_classes[y_pred[0]])
y_pred = svm.predict([data2])
print('data :', data2, ', classification:', dog_classes[y_pred[0]])

### Using SVM for larger datasets

In [ ]:
import pandas as pd
import numpy as np

data_loc = 'https://github.com/dongupak/DataML/raw/main/csv/'
df = pd.read_csv(data_loc + 'two_classes.csv')
df.tail(3)

In [ ]:
df_positive = df[df['y']==1]     # y가 1인 데이터만 추출
df_negative = df[df['y']==0]     # y가 0인 데이터만 추출
import matplotlib.pyplot as plt
plt.scatter(df_positive['x1'], df_positive['x2'], color='r')
plt.scatter(df_negative['x1'], df_negative['x2'], color='g')

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
X = df[['x1', 'x2']].to_numpy()           # x1, x2 are the input features
y = df['y']                               # y is the target label
svm = Pipeline([  # standard scaler, linear SVM classifier
                ('scaler', StandardScaler()),
                ('linearSVC', LinearSVC(C=1, loss='hinge', dual='auto'))
])
svm.fit(X, y)     # learn a svm model with X, y inputs

In [ ]:
svm.predict([[0.12, 0.56], [-4, 40], [0, 40], [5,20]])

In [ ]:
svm.named_steps['linearSVC'].coef_

### kernel SVM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
X_xor = np.random.randn(200, 2)
y_xor = np.logical_xor(X_xor[:, 0] > 0, X_xor[:, 1] > 0)
y_xor = np.where(y_xor, 1, 0)
plt.scatter(X_xor[y_xor == 1, 0], X_xor[y_xor == 1, 1],
            c='b', marker='o', label='class 1', s=50)
plt.scatter(X_xor[y_xor == 0, 0], X_xor[y_xor == 0, 1],
            c='r', marker='s', label='class 0', s=50)
plt.legend()
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("XOR")

### Visualization of the SVM results

In [ ]:
import matplotlib as mpl

def plot_xor(X, y, model, title, xmin=-3, xmax=3, ymin=-3, ymax=3):
    XX, YY = np.meshgrid(np.arange(xmin, xmax, (xmax-xmin)/1000),
                         np.arange(ymin, ymax, (ymax-ymin)/1000))
    ZZ = np.reshape(model.predict(
        np.array([XX.ravel(), YY.ravel()]).T), XX.shape)
    plt.contourf(XX, YY, ZZ, cmap=mpl.cm.Paired_r, alpha=0.5)
    plt.scatter(X[y == 1, 0], X[y == 1, 1], c='b',
                marker='o', label='class 1', s=50)
    plt.scatter(X[y == 0, 0], X[y == 0, 1], c='r',
                marker='s', label='class 0', s=50)
    plt.xlim(xmin, xmax)
    plt.ylim(ymin, ymax)
    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

linsvc = SVC(kernel="linear").fit(X_xor, y_xor)
polysvc = SVC(kernel="poly", degree=2, gamma=1, coef0=0).fit(X_xor, y_xor)
rbfsvc = SVC(kernel="rbf").fit(X_xor, y_xor)

print('linear SVM accuracy: ', accuracy_score(y_xor, linsvc.predict(X_xor)))
print('polynomial SVM accuracy: ', accuracy_score(y_xor, polysvc.predict(X_xor)))
print('rbf SVM accuracy: ', accuracy_score(y_xor, rbfsvc.predict(X_xor)))

In [ ]:
plt.figure(figsize=(8, 12))
plt.subplot(311)
plot_xor(X_xor, y_xor, linsvc, "Linear SVM")
plt.subplot(312)
plot_xor(X_xor, y_xor, rbfsvc, "rbf SVM")
plt.subplot(313)
plot_xor(X_xor, y_xor, polysvc, "polynomial SVM")
plt.tight_layout()